In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import faiss
import time
from pathlib import Path

faiss.omp_set_num_threads(1)
torch.set_num_threads(1)

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
CKPT_PATH = Path.home() / "projects" / "recsys" / "checkpoints" / "two_tower.pt"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")


class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)


# load two-tower checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
n_users = ckpt["n_users"]
n_movies = ckpt["n_movies"]
emb_dim = ckpt["config"]["emb_dim"]
user_to_idx = ckpt["user_to_idx"]
movie_to_idx = ckpt["movie_to_idx"]

model = TwoTower(n_users, n_movies, dim=emb_dim).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"two-tower loaded: {n_users:,} users × {n_movies:,} movies × {emb_dim} dim")

# precompute all item vectors + faiss index (we'll need these for negative scoring)
with torch.no_grad():
    all_movies_t = torch.arange(n_movies, dtype=torch.long, device=device)
    item_vecs = model.encode_item(all_movies_t).cpu().numpy().astype(np.float32)
print(f"item vectors: {item_vecs.shape}")

index = faiss.IndexFlatIP(emb_dim)
index.add(item_vecs)
print(f"faiss index built: {index.ntotal:,} items")

# load + split ratings, same as before
ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df = ratings[ratings["timestamp"] < cutoff_ts].copy()
val_df = ratings[ratings["timestamp"] >= cutoff_ts].copy()

for df in (train_df, val_df):
    df["user_idx"] = df["userId"].map(user_to_idx)
    df["movie_idx"] = df["movieId"].map(movie_to_idx)
    df.dropna(subset=["user_idx", "movie_idx"], inplace=True)
    df["user_idx"] = df["user_idx"].astype(np.int32)
    df["movie_idx"] = df["movie_idx"].astype(np.int32)

LIKE_THRESHOLD = 4.0
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD].reset_index(drop=True)
print(f"\ntrain rows: {len(train_df):,}")
print(f"val rows:   {len(val_df):,}")
print(f"train positives: {len(train_pos):,}")

device: mps
two-tower loaded: 150,330 users × 45,058 movies × 64 dim
item vectors: (45058, 64)
faiss index built: 45,058 items

train rows: 22,454,535
val rows:   454,141
train positives: 11,177,787


In [2]:
RANKER_SEED = 42
N_POS_SAMPLE = 1_000_000   # 1m positives, 5 negs each = 6m total rows
N_NEG_PER_POS = 5
NEG_ALPHA = 0.75

rng = np.random.RandomState(RANKER_SEED)

# subsample positives
print("subsampling 1m positives...")
sample_idx = rng.choice(len(train_pos), size=N_POS_SAMPLE, replace=False)
pos_sample = train_pos.iloc[sample_idx][["user_idx", "movie_idx"]].reset_index(drop=True)
print(f"  pos_sample: {len(pos_sample):,} rows")

# user_to_seen for rejection
print("\nbuilding user_to_seen lookup...")
t0 = time.time()
user_to_seen = {}
for uidx, group in train_df.groupby("user_idx"):
    user_to_seen[int(uidx)] = set(group["movie_idx"].values.tolist())
print(f"  done in {time.time()-t0:.1f}s")

# popularity sampling distribution
print("\ncomputing popularity sampling distribution...")
pop_count = np.zeros(n_movies, dtype=np.float64)
counts = train_df["movie_idx"].value_counts()
pop_count[counts.index.values] = counts.values
pop_weighted = pop_count ** NEG_ALPHA
sampling_probs = pop_weighted / pop_weighted.sum()

# pre-sample a big pool of negative candidates
pool_size = N_POS_SAMPLE * N_NEG_PER_POS * 3   # 3x overage to cover rejection
print(f"pre-sampling negative pool of {pool_size:,}...")
t0 = time.time()
neg_pool = rng.choice(n_movies, size=pool_size, p=sampling_probs).astype(np.int32)
print(f"  done in {time.time()-t0:.1f}s")

# build (user, movie, label) triples
print(f"\ngenerating {N_NEG_PER_POS} negatives per positive...")
t0 = time.time()
rows_user = []
rows_movie = []
rows_label = []
pool_idx = 0

for i in range(N_POS_SAMPLE):
    u = int(pos_sample["user_idx"].iloc[i])
    pos_m = int(pos_sample["movie_idx"].iloc[i])
    seen = user_to_seen[u]
    
    # add the positive
    rows_user.append(u)
    rows_movie.append(pos_m)
    rows_label.append(1)
    
    # sample N negatives
    sampled = 0
    while sampled < N_NEG_PER_POS:
        neg_m = int(neg_pool[pool_idx % len(neg_pool)])
        pool_idx += 1
        if neg_m not in seen:
            rows_user.append(u)
            rows_movie.append(neg_m)
            rows_label.append(0)
            sampled += 1
    
    if (i + 1) % 100_000 == 0:
        print(f"  {i+1:,} positives processed | {time.time()-t0:.1f}s")

print(f"  done in {time.time()-t0:.1f}s\n")

ranker_df = pd.DataFrame({
    "user_idx": np.array(rows_user, dtype=np.int32),
    "movie_idx": np.array(rows_movie, dtype=np.int32),
    "label": np.array(rows_label, dtype=np.int8),
})
print(f"ranker_df: {ranker_df.shape}")
print(f"  positives: {(ranker_df['label'] == 1).sum():,}")
print(f"  negatives: {(ranker_df['label'] == 0).sum():,}")
print(f"  pos ratio: {(ranker_df['label'] == 1).mean():.3f}")
print(f"\nsample rows:")
print(ranker_df.head(8))

subsampling 1m positives...
  pos_sample: 1,000,000 rows

building user_to_seen lookup...
  done in 4.5s

computing popularity sampling distribution...
pre-sampling negative pool of 15,000,000...
  done in 1.2s

generating 5 negatives per positive...
  100,000 positives processed | 1.6s
  200,000 positives processed | 3.1s
  300,000 positives processed | 5.1s
  400,000 positives processed | 6.6s
  500,000 positives processed | 8.2s
  600,000 positives processed | 9.7s
  700,000 positives processed | 11.3s
  800,000 positives processed | 12.9s
  900,000 positives processed | 14.4s
  1,000,000 positives processed | 16.0s
  done in 16.0s

ranker_df: (6000000, 3)
  positives: 1,000,000
  negatives: 5,000,000
  pos ratio: 0.167

sample rows:
   user_idx  movie_idx  label
0    107306       2876      1
1    107306      11290      0
2    107306        827      0
3    107306        568      0
4    107306       3864      0
5    107306       4523      0
6    124069        564      1
7    124069  

In [3]:
print("loading feature tables...")
user_features = pd.read_parquet(PARQUET_DIR / "user_features.parquet")
movie_features = pd.read_parquet(PARQUET_DIR / "movie_features.parquet")

# the feature tables use the ORIGINAL movielens userId/movieId, not our dense idx.
# we need to remap. build reverse maps once.
idx_to_user_id = {i: u for u, i in user_to_idx.items()}
idx_to_movie_id = {i: m for m, i in movie_to_idx.items()}

# add original ids back to feature tables (use original cols)
print(f"  user_features: {user_features.shape}, columns: {user_features.columns.tolist()}")
print(f"  movie_features: {movie_features.shape}, columns: {movie_features.columns.tolist()}")

# join user features: keyed on userId (original)
# add user_idx as the join key
user_features = user_features.copy()
user_features["user_idx"] = user_features["userId"].map(user_to_idx)
user_features = user_features.dropna(subset=["user_idx"])
user_features["user_idx"] = user_features["user_idx"].astype(np.int32)

# similarly for movies
movie_features = movie_features.copy()
movie_features["movie_idx"] = movie_features["movieId"].map(movie_to_idx)
movie_features = movie_features.dropna(subset=["movie_idx"])
movie_features["movie_idx"] = movie_features["movie_idx"].astype(np.int32)

# select feature columns (drop ids — we want only numeric features)
user_feat_cols = ["num_ratings", "mean_rating", "std_rating", "min_rating", "max_rating",
                  "active_seconds", "pct_high", "pct_low"]
movie_feat_cols = ["num_ratings", "num_unique_users", "mean_rating", "std_rating",
                   "pct_high", "pct_low", "smoothed_mean"]

# rename to avoid collision (both have "num_ratings", "mean_rating", etc.)
user_feats = user_features[["user_idx"] + user_feat_cols].copy()
user_feats.columns = ["user_idx"] + [f"u_{c}" for c in user_feat_cols]

movie_feats = movie_features[["movie_idx"] + movie_feat_cols].copy()
movie_feats.columns = ["movie_idx"] + [f"m_{c}" for c in movie_feat_cols]

print(f"\nuser feats (renamed): {user_feats.shape}, columns: {user_feats.columns.tolist()}")
print(f"movie feats (renamed): {movie_feats.shape}, columns: {movie_feats.columns.tolist()}")

# join everything in
print("\njoining features into ranker_df...")
t0 = time.time()
df = ranker_df.merge(user_feats, on="user_idx", how="left")
df = df.merge(movie_feats, on="movie_idx", how="left")
print(f"  done in {time.time()-t0:.1f}s, shape: {df.shape}")
print(f"  nulls after join: {df.isnull().sum().sum()}")
print(f"\nfeature columns: {[c for c in df.columns if c not in ['user_idx', 'movie_idx', 'label']]}")

loading feature tables...
  user_features: (162265, 11), columns: ['userId', 'num_ratings', 'mean_rating', 'std_rating', 'min_rating', 'max_rating', 'active_seconds', 'pct_high', 'pct_low', 'first_rating_ts', 'last_rating_ts']
  movie_features: (58616, 10), columns: ['movieId', 'num_ratings', 'num_unique_users', 'mean_rating', 'std_rating', 'first_rating_ts', 'last_rating_ts', 'pct_high', 'pct_low', 'smoothed_mean']

user feats (renamed): (150330, 9), columns: ['user_idx', 'u_num_ratings', 'u_mean_rating', 'u_std_rating', 'u_min_rating', 'u_max_rating', 'u_active_seconds', 'u_pct_high', 'u_pct_low']
movie feats (renamed): (45058, 8), columns: ['movie_idx', 'm_num_ratings', 'm_num_unique_users', 'm_mean_rating', 'm_std_rating', 'm_pct_high', 'm_pct_low', 'm_smoothed_mean']

joining features into ranker_df...
  done in 0.3s, shape: (6000000, 18)
  nulls after join: 0

feature columns: ['u_num_ratings', 'u_mean_rating', 'u_std_rating', 'u_min_rating', 'u_max_rating', 'u_active_seconds', '

In [4]:
print("computing two-tower scores for all 6m (user, movie) pairs...")
t0 = time.time()

# extract unique (user, movie) pairs to score
# we have 6m rows but many duplicate users (each user has 6 rows) and many duplicate movies
# the easiest path: just score every row, batched on mps

BATCH = 200_000
n_rows = len(df)
scores = np.empty(n_rows, dtype=np.float32)

user_idx_t = torch.from_numpy(df["user_idx"].values.astype(np.int64))
movie_idx_t = torch.from_numpy(df["movie_idx"].values.astype(np.int64))

with torch.no_grad():
    for start in range(0, n_rows, BATCH):
        end = min(start + BATCH, n_rows)
        u_batch = user_idx_t[start:end].to(device)
        m_batch = movie_idx_t[start:end].to(device)
        uv = model.encode_user(u_batch)
        iv = model.encode_item(m_batch)
        s = (uv * iv).sum(dim=1).cpu().numpy()
        scores[start:end] = s
        if (start // BATCH) % 5 == 0:
            print(f"  scored {end:>9,} / {n_rows:,} ({100*end/n_rows:.1f}%)  {time.time()-t0:.1f}s")

df["tt_score"] = scores
print(f"\ndone in {time.time()-t0:.1f}s\n")

print("two-tower score stats by label:")
print(df.groupby("label")["tt_score"].describe().round(4))

computing two-tower scores for all 6m (user, movie) pairs...
  scored   200,000 / 6,000,000 (3.3%)  0.4s
  scored 1,200,000 / 6,000,000 (20.0%)  0.4s
  scored 2,200,000 / 6,000,000 (36.7%)  0.4s
  scored 3,200,000 / 6,000,000 (53.3%)  0.5s
  scored 4,200,000 / 6,000,000 (70.0%)  0.5s
  scored 5,200,000 / 6,000,000 (86.7%)  0.5s

done in 0.5s

two-tower score stats by label:
           count    mean     std     min     25%     50%     75%     max
label                                                                   
0      5000000.0  0.0658  0.4650 -5.3791 -0.1203 -0.0029  0.1888  5.5142
1      1000000.0  0.8023  0.7612 -2.6376  0.2153  0.6376  1.2164  8.7149


the two-tower score is the strongest single feature: positives have mean 0.80, negatives 0.07. but the distributions overlap, so popularity-based or content-based features must add complementary signal to lift recall further.

In [5]:
# random 80/20 split of the 6m rows
print("splitting ranker dataset 80/20...")
n_rows = len(df)
perm = rng.permutation(n_rows)
train_n = int(n_rows * 0.8)
train_perm = perm[:train_n]
eval_perm = perm[train_n:]

train_ranker = df.iloc[train_perm].reset_index(drop=True)
eval_ranker = df.iloc[eval_perm].reset_index(drop=True)
print(f"  train: {len(train_ranker):,}")
print(f"  eval:  {len(eval_ranker):,}")
print(f"  train pos ratio: {train_ranker['label'].mean():.4f}")
print(f"  eval pos ratio:  {eval_ranker['label'].mean():.4f}")

# save to disk so tomorrow's training doesn't need to redo all this work
out_dir = PARQUET_DIR / "ranker"
out_dir.mkdir(exist_ok=True)

t0 = time.time()
train_ranker.to_parquet(out_dir / "train.parquet", compression="zstd")
eval_ranker.to_parquet(out_dir / "eval.parquet", compression="zstd")
print(f"\nsaved to {out_dir} in {time.time()-t0:.1f}s")
print(f"  train.parquet: {(out_dir/'train.parquet').stat().st_size/1e6:.1f} mb")
print(f"  eval.parquet:  {(out_dir/'eval.parquet').stat().st_size/1e6:.1f} mb")

print(f"\nfinal columns: {df.columns.tolist()}")

splitting ranker dataset 80/20...
  train: 4,800,000
  eval:  1,200,000
  train pos ratio: 0.1666
  eval pos ratio:  0.1668

saved to /Users/nitishpatil/projects/recsys/data/parquet/ranker in 1.2s
  train.parquet: 159.9 mb
  eval.parquet:  40.8 mb

final columns: ['user_idx', 'movie_idx', 'label', 'u_num_ratings', 'u_mean_rating', 'u_std_rating', 'u_min_rating', 'u_max_rating', 'u_active_seconds', 'u_pct_high', 'u_pct_low', 'm_num_ratings', 'm_num_unique_users', 'm_mean_rating', 'm_std_rating', 'm_pct_high', 'm_pct_low', 'm_smoothed_mean', 'tt_score']
